# 第 01 课作业：使用全 A 股真实行情的股票研究智能体

**目标：** 根据用户条件在沪、深、北 A 股范围筛选，形成有数据依据的研究候选，并解释覆盖、来源、时间与未知项。

本作业沿用课程四步：模型客户端 → 工具 → 智能体 → 流式回复。
默认通过网络读取 A 股证券名录与行情。先载入完整证券范围，再批量取得行情并核对覆盖；工具和网页分页显示，筛选仍作用于完整范围。

全市场范围不等于所有指标：来源 PE 未明确口径，使用 pe_ratio，不宣称是 TTM；股息率和行业仍保留缺失值。
as_of 是来源报价时间，fetched_at 是程序抓取时间；批量行情具有抓取起止时间，不保证同一瞬间或交易所级实时。


## 1. 环境准备：连接模型

先在仓库根目录安装 `requirements.txt`，按 `.env.example` 配好 `LLM_BASE_URL`、`LLM_API_KEY`、`LLM_MODEL`，选择项目的 Python 内核。本单元沿用课程的环境加载和 `ChatOpenAI` 写法；额外向上定位仓库根目录读取课程共用的配置，并从当前中文作业目录导入 stock_agent。只打印初始化状态，不打印密钥；缺少配置时会明确报错。

In [1]:
import json
import os

from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv(find_dotenv())

missing = [name for name in ("LLM_BASE_URL", "LLM_API_KEY", "LLM_MODEL") if not os.environ.get(name)]
if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Copy .env.example to .env in the repository root and fill them in."
    )

llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ["LLM_API_KEY"],
    extra_body=json.loads(os.environ.get("LLM_EXTRA_BODY") or "null"),
)
print(f"Model client ready: {os.environ['LLM_MODEL']} @ {os.environ['LLM_BASE_URL']}")

Model client ready: deepseek-v4-pro @ https://api.deepseek.com/v1


## 2. 数据层：取得完整范围并核对覆盖

create_provider() 默认使用真实全 A 股数据；只有显式设置 STOCK_SCOPE=watchlist，才使用 STOCK_SYMBOLS 指定的关注范围。单独设置 STOCK_SYMBOLS 不会缩小默认的全 A 股范围。
get_market_summary 先读取完整证券范围，报告实际证券数、交易所覆盖和抓取起止时间。完整加载后，只打印前 5 条记录，避免数千行输出。
数据失败或覆盖不完整会明确报告，不会悄悄换成模拟数据。短时间重复查询共用内存缓存。

每条记录保留来源报价时间、抓取时间、价格和市盈率口径；缺失股息率是 None，不是零股息。
离线学习需明确设置 STOCK_DATA_MODE=demo，程序会显示模拟范围。


In [2]:
from stock_agent.data import create_provider
from stock_agent.queries import market_summary, stock_page

provider = create_provider()
summary = market_summary(provider)
print("实际数据范围与覆盖：", json.dumps(summary, ensure_ascii=False, indent=2))
preview = stock_page(provider, limit=5)
print(f"完整范围共 {preview['universe_count']} 只；以下只展示前 {len(preview['items'])} 只。")
for row in preview["items"]:
    print(json.dumps(row, ensure_ascii=False))


实际数据范围与覆盖： {
  "data_mode": "live",
  "scope": "all_a",
  "scope_label": "全 A 股（沪深京，按来源证券目录）",
  "source": "腾讯财经公开行情接口",
  "source_url": "https://qt.gtimg.cn/",
  "universe_source": "腾讯财经沪深京 A 股公开目录",
  "universe": {
    "source": "腾讯财经沪深京 A 股公开目录",
    "source_url": "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList?board_code=aStock",
    "page_url": "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList",
    "fetched_at": "2026-09-15T09:09:18.295560+00:00",
    "count": 5561,
    "exchange_counts": {
      "sh": 2317,
      "sz": 2901,
      "bj": 343
    },
    "cache_ttl_seconds": 86400,
    "cache_origin": "disk",
    "freshness_note": "目录缓存最多保留 24 小时，新增上市股票可能在下一次目录刷新后出现。",
    "coverage_basis": "来源 aStock 节点全部分页，验证数量、唯一代码和沪深京覆盖，再明确排除已核实的存托凭证；并非交易所独立核验。",
    "source_count": 5562,
    "source_exchange_counts": {
      "sh": 2318,
      "sz": 2901,
      "bj": 343
    },
    "excluded_instruments": [
      {
        "symbol": "689009",
        "name": 

## 3. 四个工具：总览、分页浏览、详情与完整范围筛选

get_market_summary 查询总范围和覆盖；list_stocks 分页浏览；get_stock_details 按代码查询详情；screen_stocks 先在完整范围应用条件，再分页返回匹配结果。
分页只控制展示条数，不改变筛选范围。筛选结果报告 universe_count（总范围）、total（匹配总数）、items（本页），按代码排序不代表投资排名。
当前真实源支持来源 PE 条件；未确认的股息率或行业条件会返回 unsupported_metric，不能静默删除。
run_data_tool 将已知数据错误转为可见工具结果，handle_tool_error=True 允许模型解释原因。
每个工具都保留类型注解和一句话 docstring。


In [3]:
from langchain.tools import tool
from stock_agent.agent import run_data_tool
from stock_agent.queries import market_summary, stock_page


@tool
def get_market_summary() -> dict:
    """加载完整数据范围，返回实际证券数、沪深北覆盖、来源和抓取时间。"""
    return run_data_tool(market_summary, provider)


@tool
def list_stocks(limit: int = 20, offset: int = 0) -> list[dict]:
    """分页浏览股票；limit 为 1—100，offset 从 0 开始，分页不限制筛选范围。"""
    return run_data_tool(stock_page, provider, limit=limit, offset=offset)["items"]


@tool
def get_stock_details(symbol: str) -> dict:
    """按 A 股代码查询来源数据；保留缺失字段，不能预测未来涨跌。"""
    return run_data_tool(provider.get_stock_details, symbol)


@tool
def screen_stocks(
    max_pe: float,
    min_dividend_yield_pct: float | None = None,
    sector: str | None = None,
    limit: int = 10,
    offset: int = 0,
) -> dict:
    """先筛完整范围再按代码分页；返回匹配总数、总范围与覆盖，指标不可用则报错。"""
    return run_data_tool(
        stock_page, provider, limit=limit, offset=offset, max_pe=max_pe,
        min_dividend_yield_pct=min_dividend_yield_pct, sector=sector,
    )


tools = [get_market_summary, list_stocks, get_stock_details, screen_stocks]
for item in tools:
    item.handle_tool_error = True
print("工具已注册：", [item.name for item in tools])


工具已注册： ['get_market_summary', 'list_stocks', 'get_stock_details', 'screen_stocks']


## 4. 智能体：按实际数据源解释结果

用 create_agent 组装模型和工具。build_system_prompt(provider) 把同一数据源的模式和能力写入提示词，避免真实数据被描述成模拟数据。
提示词要求先查询再解释，区分报价时间与抓取时间，保留单位与指标口径，并明确未知信息。
提示词是行为约束，后续软件还需要针对报告数字、单位和来源做程序核验。


In [4]:
from langchain.agents import create_agent
from stock_agent.agent import build_system_prompt, content_text

system_prompt = build_system_prompt(provider)
print(system_prompt)
agent = create_agent(llm, tools=tools, system_prompt=system_prompt)
RUN_CONFIG = {"recursion_limit": 20}


你是中文 A 股研究助手，帮助用户从明确条件形成可核查的研究候选。
股票事实必须来自本轮工具返回，不能用记忆补充价格、财报、新闻、行业或分红。
1. 全市场筛选前先用 get_market_summary 取得实际证券总数、沪深北覆盖和抓取时间；
   以 scope 与 coverage 为准描述范围，all_a 才是全 A 股，watchlist 是指定范围。
   list_stocks 只分页浏览，不限制筛选范围；screen_stocks 先筛完整范围，再分页返回。
   回答必须区分 universe_count、符合条件的 total 与本页 items 数，不把本页当全市场。
   代码升序只是稳定展示顺序，不代表投资排名；推荐或比较前用 get_stock_details 查详情。
   coverage.complete 为 false 时不得宣称完成全市场筛选；同时说明 screening_exclusions。
2. 以 data_mode、source、source_url、as_of、fetched_at 和 quote_status 说明数据状态。
   live 表示外部数据源实际返回，不保证交易所级实时；as_of 是来源报价时间，
   fetched_at 是程序获取时间。demo 才是人工模拟，必须标注，不可当作真实行情。
3. pe_ratio 是来源提供的市盈率。仅当 pe_basis 明确支持时才能称为 TTM；
   当前腾讯个股接口未明确该 PE 口径，不能把它当作 pe_ttm 或动态市盈率。
   数值与 metric_units 原样匹配，不自行换算、推断低估、安全边际或未来收益。
4. None、missing_fields 和 unsupported_metric 表示未知或不支持，不能当成零。
   行业字段缺失时必须写“行业未知”，即使熟悉该公司，也不得根据公司名称或常识补写行业。
   价格、PB、涨跌幅可以展示比较，但当前 screen_stocks 只支持已实现的 PE 筛选，不能承诺支持其他条件。
   用户指定股息率、行业等不可用条件时，明确无法完成这项筛选，询问是否移除条件；
   未经用户允许不得放宽。没有条件时询问，或说明所用条件。
5. 工具返回错误时按错误内容说明；区分接口不可用、覆

## 5. 普通调用：全范围筛选并展示完整消息历史

先查询全市场覆盖，再筛选来源市盈率不超过 25 的股票，只展示按代码排序的前 3 个匹配并查询这 3 个详情。
明确区分完整证券数、匹配总数、本页展示数；不把前 3 只称为最优投资排名。
遍历 result["messages"]，打印全部可见用户、模型与工具消息，包含工具参数和对应调用 ID。
末尾核对所有工具调用都有返回结果；工具分页保持消息历史可读。


In [5]:
def print_history(run_result: dict) -> None:
    """打印用户、模型、工具的完整可见消息及工具调用信息。"""
    for index, message in enumerate(run_result["messages"], start=1):
        print(f"\n--- 消息 {index} | 类型: {message.type} | 名称: {getattr(message, 'name', None)} ---")
        print("内容:", message.content)
        if getattr(message, "tool_calls", None):
            print("工具调用:", json.dumps(message.tool_calls, ensure_ascii=False, indent=2))
        if getattr(message, "tool_call_id", None):
            print("工具结果对应调用 ID:", message.tool_call_id)


question = (
    "请先调用 get_market_summary 核对全 A 股的数据覆盖，再用 screen_stocks 在完整范围内"
    "筛选市盈率不超过 25 的候选，设置 limit=3、offset=0，只展示按代码排序的前 3 只。"
    "然后分别用 get_stock_details 查询这 3 个代码。请简短说明总范围、匹配总数、展示数，"
    "代码、名称、筛选依据、市盈率口径、来源、报价时间、抓取时间和未知项。"
    "代码顺序不代表投资排名；不添加股息率或行业条件。若数据范围不是 all_a，明确说明实际范围。"
)
result = agent.invoke(
    {"messages": [{"role": "user", "content": question}]},
    config=RUN_CONFIG,
)
print_history(result)

tool_call_ids = {
    call["id"]
    for message in result["messages"]
    for call in getattr(message, "tool_calls", [])
}
tool_result_ids = {
    message.tool_call_id for message in result["messages"] if message.type == "tool"
}
assert tool_call_ids, "本次模型未调用工具，请检查模型是否支持工具调用。"
assert tool_call_ids <= tool_result_ids, "存在没有返回结果的工具调用。"
print("\n工具调用与结果检查通过。")



--- 消息 1 | 类型: human | 名称: None ---
内容: 请先调用 get_market_summary 核对全 A 股的数据覆盖，再用 screen_stocks 在完整范围内筛选市盈率不超过 25 的候选，设置 limit=3、offset=0，只展示按代码排序的前 3 只。然后分别用 get_stock_details 查询这 3 个代码。请简短说明总范围、匹配总数、展示数，代码、名称、筛选依据、市盈率口径、来源、报价时间、抓取时间和未知项。代码顺序不代表投资排名；不添加股息率或行业条件。若数据范围不是 all_a，明确说明实际范围。

--- 消息 2 | 类型: ai | 名称: None ---
内容: 我先核对全市场数据覆盖，再执行筛选。
工具调用: [
  {
    "name": "get_market_summary",
    "args": {},
    "id": "call_00_8Q8Ajl7ewwpDXvu3cX3d3618",
    "type": "tool_call"
  }
]

--- 消息 3 | 类型: tool | 名称: get_market_summary ---
内容: {"data_mode": "live", "scope": "all_a", "scope_label": "全 A 股（沪深京，按来源证券目录）", "source": "腾讯财经公开行情接口", "source_url": "https://qt.gtimg.cn/", "universe_source": "腾讯财经沪深京 A 股公开目录", "universe": {"source": "腾讯财经沪深京 A 股公开目录", "source_url": "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList?board_code=aStock", "page_url": "https://proxy.finance.qq.com/cgi/cgi-bin/rank/hs/getBoardRankList", "fetched_at": "2026-09-15T09:09:18.295560+00:00", "count": 5561,

## 6. 逐块流式比较

astream(..., stream_mode="messages") 在生成过程中产出内容，只打印 model 节点的文本。
比较两只股票的真实价格和来源 PE，说明时间和口径；没有行业、财报或股息率数据时不能自行补造。
这里是新的独立调用，不假设自动记住上一轮。


In [6]:
streamed_parts = []
async for token, metadata in agent.astream(
    {"messages": [{
        "role": "user",
        "content": (
            "请分别查询 600036 和 600900 的来源数据，简短比较价格和市盈率，"
            "注明市盈率口径、报价时间和来源，并说明为什么这些数据不能确定未来收益。行业字段缺失，请写行业未知，不要凭公司名称或常识补充行业。"
        ),
    }]},
    config=RUN_CONFIG,
    stream_mode="messages",
):
    if metadata.get("langgraph_node") == "model":
        text = content_text(token.content)
        if text:
            streamed_parts.append(text)
            print(text, end="", flush=True)
print()
assert streamed_parts, "模型没有返回流式文本，请检查服务商是否支持流式输出。"
print(f"\n收到 {len(streamed_parts)} 个模型文本块。")


我来分别查询这两只股票的来源数据。## 查询结果比较

| 项目 | 600036 招商银行 | 600900 长江电力 |
|---|---|---|
| 价格 | 40.92 元/股 | 28.46 元/股 |
| 市盈率（PE） | 6.8 倍 | 19.24 倍 |
| 市净率（PB） | 0.9 倍 | 3.22 倍 |
| 当日涨跌幅 | -0.44% | -0.14% |
| 行业 | 行业未知 | 行业未知 |
| 报价时间（as_of） | 2026-09-16 15:51:56（+08:00） | 2026-09-16 15:51:57（+08:00） |

**PE 口径说明**：腾讯财经个股接口返回的 `pe_basis` 为“来源未明确 PE 口径”，因此**不能称为 TTM 或动态市盈率**。两条记录 `pe_ttm` 均为 null（未知），仅能按“来源提供的市盈率”数值展示。

**数据来源与状态**：均为 `live` 模式（腾讯财经公开行情接口实际返回，https://qt.gtimg.cn/ ），`quote_status` 为 `current`（当前可用），报价时间为来源返回的行情时间，不保证是逐笔成交时间。

**简短比较**：招商银行市盈率（6.8 倍）明显低于长江电力（19.24 倍），市净率也显著更低（0.9 vs 3.22）；但长江电力价格（28.46 元）低于招商银行（40.92 元）。行业字段两条记录均缺失，故写“行业未知”，未凭公司名称或常识补充。

## 为什么不能确定未来收益

1. **低市盈率、低市净率是已发生的历史数据**，只反映过去和当前市场价格与财务指标的关系，不代表未来股价会因此上涨或收益兑现。
2. **PE 口径未确认**，无法确定其计算基于何种盈利（静态、动态、TTM 等），不同口径下数值含义不同，不能据此下“低估/安全边际”等结论。
3. 来源仅提供价格、PE、PB、涨跌幅，**不提供未来涨跌预测**；公司风险资料 `risk_notes` 为空，工具没有关于这两家公司的风险资料，不能补造。
4. 这些数据仅支持当前状态下的研究比较，不构成买入建议或收益保证。

收到 557 个模型文本块。


## 7. 对照实验：直接回答与缺失指标

问候不需要行情数据；要求股息率筛选及精确预测则超出当前能力。
两个问题各自调用，打印完整历史和实际工具名。对缺失指标要明确说明，不能未经允许删除用户条件，也不能承诺明日涨幅。


In [7]:
probe_results = {}
for label, prompt in {
    "问候": "你好，请用一句话介绍你能做什么，不需要查询股票。",
    "数据不足": (
        "请实际调用 screen_stocks，按市盈率不超过 25 且股息率至少 3% 筛选；"
        "不允许删除股息率条件，并告诉我候选明天准确会涨百分之几。"
    ),
}.items():
    print(f"\n========== {label} ==========")
    probe = agent.invoke(
        {"messages": [{"role": "user", "content": prompt}]},
        config=RUN_CONFIG,
    )
    probe_results[label] = probe
    print_history(probe)
    calls = [
        call["name"]
        for message in probe["messages"]
        for call in getattr(message, "tool_calls", [])
    ]
    print("本次实际调用的工具:", calls or "无")



========== 问候 ==========

--- 消息 1 | 类型: human | 名称: None ---
内容: 你好，请用一句话介绍你能做什么，不需要查询股票。

--- 消息 2 | 类型: ai | 名称: None ---
内容: 你好，我可以基于腾讯财经公开行情接口，在全 A 股（沪深京）范围内按市盈率等指标进行筛选、查看个股详情与行情来源，并形成有数据支撑的研究候选列表，但不能预测涨跌或提供交易建议。
本次实际调用的工具: 无

========== 数据不足 ==========

--- 消息 1 | 类型: human | 名称: None ---
内容: 请实际调用 screen_stocks，按市盈率不超过 25 且股息率至少 3% 筛选；不允许删除股息率条件，并告诉我候选明天准确会涨百分之几。

--- 消息 2 | 类型: ai | 名称: None ---
内容: 我将调用 `screen_stocks` 按您指定的条件筛选。需要说明：股息率条件当前接口并未支持（工具能力中仅支持 `max_pe` 筛选），同时工具不提供任何未来涨跌预测。
工具调用: [
  {
    "name": "screen_stocks",
    "args": {
      "max_pe": 25,
      "min_dividend_yield_pct": 3
    },
    "id": "call_00_7bBmaTyQ3hdtLVGunrCm8427",
    "type": "tool_call"
  }
]

--- 消息 3 | 类型: tool | 名称: screen_stocks ---
内容: {"error": "unsupported_metric", "message": "当前未接入已核实的股息率，不能按股息率筛选全 A 股。"}
工具结果对应调用 ID: call_00_7bBmaTyQ3hdtLVGunrCm8427

--- 消息 4 | 类型: ai | 名称: None ---
内容: 我无法完成这项筛选，原因如下：

## 1. 筛选结果：无法执行

工具返回错误 `unsupported_metric`，内容为：**"当前未接入已核实的股息率，不能按股息率筛选全 A 股。"**


### 三个问题的回答

当我要求按市盈率不超过 25 筛选股票并核对候选详情时，模型调用了 `get_market_summary` 查看市场范围、`screen_stocks` 执行筛选，以及三次 `get_stock_details` 查询前三只候选的具体数据。
当我只向它问好并请它用一句话介绍能做什么时，模型没有调用工具，而是直接介绍了自己的能力，输出中的“本次实际调用的工具”为“无”。
当我要求同时满足“市盈率不超过 25、股息率至少 3%”时，`screen_stocks` 返回了 `unsupported_metric`，说明当前没有接入可核实的股息率数据。
模型随后解释了为什么无法完成这项联合筛选，并说明工具不支持明日精确涨幅预测，保留了我要求的股息率条件，也没有编造缺失数据或候选结果。


## 8. 整个项目的反思

通过这个项目，我把课堂中的旅行智能体改成了面向 A 股的研究助手，走通了连接模型、封装工具、查询真实数据、筛选和解释结果的流程。
我认识到，模型主要负责理解问题和组织回答，股票数据应由接口提供，筛选规则应由代码执行，不能让模型凭印象补充数字。
从几只关注股票扩展到全 A 股后，我发现除了让程序跑通，还需要核对证券范围、数据来源、更新时间和缺失值，并把全量筛选与分页展示分开。
打印完整消息历史和观察流式输出，让我能检查答案的依据，也提醒我工具调用成功并不代表模型的解释一定正确。
接下来我计划沿课程继续加入多轮记忆，并补齐股息率、行业、财务口径和结果核验，让这份作业逐步成为可持续使用的股票研究软件。
